# Setup and Environment Validation
**UN World Population Prospects (WPP 2024) - ASA DataQuest 2026**

---


## 0.1 Diagnóstico de Dependencias y Configuración del Sistema
Verificamos las versiones de las herramientas clave para evitar discrepancias de entorno entre colaboradores y jurados.

In [2]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

sys.path.insert(0, str(Path.cwd().parent))
from src.config import BRONZE, DOCS, SEED
from src.loaders import load_medium, load_supplement, load_notes
from src.validators import check_bronze_files_exist, generate_quality_report

print(f"Python: {sys.version.split()[0]}")
print(f"Pandas: {pd.__version__}")
print(f"NumPy:  {np.__version__}")

Python: 3.11.9
Pandas: 3.0.6
NumPy:  2.4.6


## 0.2 Auditoría de Archivos en la Capa Bronze
> **Regla de Arquitectura Medallion:** Los archivos en `data/bronze/` son de solo lectura e inmutables. Representan la fuente primaria de datos tal como fueron descargados de la División de Población de las Naciones Unidas (UN DESA).

In [3]:
bronze_audit = check_bronze_files_exist()
for name, info in bronze_audit.items():
    print(f" - {name:15s}: existe={info['exists']}, tamano={info['size_mb']} MB")

 - Medium         : existe=True, tamano=15.79 MB
 - OtherVariants  : existe=True, tamano=71.85 MB
 - Supplement     : existe=True, tamano=2.01 MB
 - Notes          : existe=True, tamano=0.0 MB


## 0.3 Previsualización Rápida del Dataset Principal (`Medium`)
La variante `Medium` es la proyección central de fecundidad, mortalidad y migración de la ONU. Abarca observaciones históricas (1950–2023) y estimaciones proyectadas (2024–2100).

In [4]:
# Muestra inicial para auditar forma y tipos de datos
df_medium_sample = load_medium(nrows=1000)

print(f"Dimensión de muestra inspeccionada: {df_medium_sample.shape[0]} filas × {df_medium_sample.shape[1]} columnas")
print(f"Uso de memoria (muestra): {df_medium_sample.memory_usage(deep=True).sum() / (1024*1024):.2f} MB")

# Vista de columnas estructurales clave
cols_preview = ['Location', 'Time', 'LocTypeName', 'TPopulation1July', 'TFR', 'LEx', 'IMR', 'PopGrowthRate']
cols_available = [c for c in cols_preview if c in df_medium_sample.columns]

display(df_medium_sample[cols_available].head())

Dimensión de muestra inspeccionada: 1000 filas × 67 columnas
Uso de memoria (muestra): 0.54 MB


,Location,Time,LocTypeName,TPopulation1July,TFR,LEx,IMR,PopGrowthRate
0,ADB region: Central and West Asia,1950,NaN,69213.424,5.9875,38.6085,207.0834,2.058
1,ADB region: Central and West Asia,1951,NaN,70682.760,5.9815,39.6435,200.9781,2.143
2,ADB region: Central and West Asia,1952,NaN,72238.716,5.9847,40.5673,196.0193,2.212
3,ADB region: Central and West Asia,1953,NaN,73876.364,5.9954,41.4405,190.7597,2.271
4,ADB region: Central and West Asia,1954,NaN,75600.454,6.0042,42.2044,185.8880,2.342


In [5]:
# Reporte rápido de calidad y completitud sobre la muestra
quality_summary = generate_quality_report(df_medium_sample)
for k, v in quality_summary.items():
    print(f" - {k:18s}: {v}")

 - n_rows            : 1000
 - n_cols            : 67
 - memory_mb         : 0.54
 - n_duplicates      : 0
 - columns_with_na   : 61
 - total_na_cells    : 8702
 - na_pct            : 12.99


## 0.4 Previsualización del Suplemento de Nacimientos y LE60
El archivo `WPP2024_Supplement_Births_and_LE60.csv` provee el desglose por sexo de los nacimientos anuales y la esperanza de vida a los 60 años ($e_{60}$), esencial para analizar sobrevida geriátrica y sex ratio al nacer.

In [6]:
# Cargar muestra del suplemento
suppl_sample = load_supplement(nrows=500)

print(f"Dimensiones del suplemento: {suppl_sample.shape}")
display(suppl_sample.head())

Dimensiones del suplemento: (500, 9)


,LocID,ISO3_code,Location,Time,BirthsMale,BirthsFemale,LE60,LE60Male,LE60Female
0,4,AFG,Afghanistan,1950,196.623,187.363,10.700,10.386,11.132
1,4,AFG,Afghanistan,1951,200.216,190.786,10.777,10.453,11.221
2,4,AFG,Afghanistan,1952,203.609,194.054,10.852,10.517,11.309
3,4,AFG,Afghanistan,1953,207.190,197.476,10.929,10.582,11.398
4,4,AFG,Afghanistan,1954,210.168,200.260,10.961,10.608,11.434


---
**Siguiente paso:** Proceder a ejecutar [`01_bronze_to_silver.ipynb`](./01_bronze_to_silver.ipynb) para ejecutar la limpieza, tipado estricto, unión de datasets y serialización a formato `.parquet`.
